In [ ]:
import re
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from analyses.utils.aggregate_data import aggregate_data


def _extract_channels(df):
    pattern = re.compile(r"\d{3,4}")
    return sorted([col for col in df.columns if pattern.search(col)])

def _wavelength_to_color(wavelengths):
    norm = mcolors.Normalize(vmin=min(wavelengths), vmax=max(wavelengths))
    colormap = cm.get_cmap("rainbow")
    return [mcolors.to_hex(colormap(norm(w))) for w in wavelengths]

def _extract_channels(df: pl.DataFrame) -> list[str]:
    pattern = re.compile(r"\d{3,4}")
    return sorted([col for col in df.columns if pattern.search(col)])

# def plot_time_series(
#     df: pl.DataFrame,
#     channels_and_colors: dict,
#     highlight_channel: str = None,
#     dtm: str = "dtm",
#     output_path: Path = Path("ae31_plot.png"),
#     figsize: tuple = (10, 5),
#     title: str = "Aethalometer AE31 hourly data (Dagoretti Corner, NRB)",
#     xlabel: str = "Date",
#     ylabel: str = "Aerosol particle absorption (expressed as ng/m3)",
#     legend_title: str = "Channels",
#     legend_loc: str = "upper right",
#     legend_fontsize: int = 8,
#     plot_type: str = "line"  # or "scatter"
# ):
#     assert plot_type in {"line", "scatter"}, "plot_type must be 'line' or 'scatter'"

#     # Convert to pandas for plotting
#     df_pd = df.select([dtm] + list(channels_and_colors.keys())).to_pandas()

#     plt.figure(figsize=figsize)
#     ax = plt.gca()

#     for ch, color in channels_and_colors.items():
#         if plot_type == "line":
#             line_width = 2.5 if ch == highlight_channel else 1.2
#             ax.plot(df_pd[dtm], df_pd[ch], label=ch, color=color, linewidth=line_width)
#         else:  # scatter
#             marker_size = 10 if ch == highlight_channel else 5
#             ax.scatter(df_pd[dtm], df_pd[ch], label=ch, color=color, s=marker_size, alpha=0.8)

#     ax.set_title(title)
#     ax.set_xlabel(xlabel)
#     ax.set_ylabel(ylabel)
#     ax.legend(
#         title=legend_title,
#         loc=legend_loc,
#         fontsize=legend_fontsize,
#         title_fontsize=legend_fontsize
#     )

#     plt.tight_layout()
#     plt.savefig(output_path, dpi=300)
#     plt.close()

#     print(f"✅ {plot_type.capitalize()} plot saved to {output_path}")
def plot_time_series(
    df: pl.DataFrame,
    channels_and_colors: dict,
    highlight_channel: str = None,
    dtm: str = "dtm",
    output_path: Path = Path("ae31_plot.png"),
    figsize: tuple = (10, 5),
    title: str = "Aethalometer AE31 hourly data (Dagoretti Corner, NRB)",
    xlabel: str = "Date",
    ylabel: str | tuple[str] | None = "ylabel",
    legend_title: str = "Channels",
    legend_loc: str = "upper right",
    legend_fontsize: int = 8,
    plot_type: str = "line",  # Options: "line" or "scatter"
    which_y_axes: tuple[int] | None = None,  # 1 = left, 2 = right
    grid: bool = False,  # Enable or disable grid
):
    assert plot_type in {"line", "scatter"}, "plot_type must be 'line' or 'scatter'"

    df_pd = df.to_pandas()
    x = df_pd[dtm]

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    # Set y-axis labels
    if isinstance(ylabel, list) and len(ylabel) == 2:
        ax1.set_ylabel(ylabel[0])
        ax2.set_ylabel(ylabel[1])
    elif isinstance(ylabel, str):
        ax1.set_ylabel(ylabel)
    elif ylabel is not None:
        raise ValueError("ylabel must be a string, a list of two strings, or None")

    # Assign all channels to axis 1 by default
    if which_y_axes is None:
        which_y_axes = [1] * len(channels_and_colors)

    if len(which_y_axes) != len(channels_and_colors):
        raise ValueError("Length of which_y_axes must match number of channels.")

    # Map channels to their axis
    ch_axis = dict(zip(channels_and_colors.keys(), which_y_axes))

    for ch, color in channels_and_colors.items():
        y = df_pd[ch]
        ax = ax1 if ch_axis[ch] == 1 else ax2

        if ch == highlight_channel:
            ax.scatter(x, y, color=color, label=ch, s=10, zorder=5)
        else:
            if plot_type == "scatter":
                ax.scatter(x, y, color=color, label=ch, s=8)
            else:
                ax.plot(x, y, color=color, label=ch, linewidth=1.0)

    ax1.set_title(title)
    ax1.set_xlabel(xlabel)

    # Grid control
    ax1.grid(grid)
    ax2.grid(False)

    # Enable horizontal (x-axis) tick marks
    ax1.tick_params(axis='x', which='both', direction='out', bottom=True, top=False, length=4)
    ax2.tick_params(axis='x', which='both', direction='out', bottom=True, top=False, length=4)

    # Merge and display legend
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        title=legend_title,
        loc=legend_loc,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    fig.autofmt_xdate()
    fig.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"✅ Time series {plot_type} plot saved to {output_path}")    

def plot_diurnal_by_wavelength(
    df: pl.DataFrame,
    channels_and_colors: dict,
    dtm: str = "dtm",
    plot_type: str = "box",  # Options: "box", "violin", "swarm"
    output_path: Path = Path("ae31_diurnal.png"),
    figsize: tuple = (10, 5),
    title: str = "Diurnal variability of aerosol absorption (Dagoretti Corner, NRB)",
    xlabel: str = "Hour of Day",
    ylabel: str = "Aerosol particle absorption (ng/m³)",
    legend_title: str = "Channels",
    legend_loc: str = "upper right",
    legend_fontsize: int = 8,
):
    assert plot_type in {"box", "violin", "swarm"}, "plot_type must be 'box', 'violin', or 'swarm'"

    # 1. Add hour column
    df = df.with_columns(
        pl.col(dtm).dt.hour().alias("hour")
    )

    # 2. Melt to long format
    melted = df.melt(
        id_vars=["hour"],
        value_vars=list(channels_and_colors.keys()),
        variable_name="channel",
        value_name="value"
    )

    # 3. Filter negative values
    melted = melted.filter(pl.col("value") >= 0)

    # 4. Convert to pandas for Seaborn
    df_plot = melted.to_pandas()

    # 5. Ensure channel order by wavelength
    channel_order = list(channels_and_colors.keys())

    # 6. Plot
    plt.figure(figsize=figsize)
    ax = plt.gca()

    if plot_type == "box":
        sns.boxplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            width=0.7,
            fliersize=2,
            ax=ax,
        )
    elif plot_type == "violin":
        sns.violinplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            linewidth=0.8,
            ax=ax,
        )
    elif plot_type == "swarm":
        sns.swarmplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            size=2,
            ax=ax,
        )

    # Final touches
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Legend
    ax.legend(
        title=legend_title,
        loc=legend_loc,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"✅ Diurnal {plot_type} plot saved to {output_path}")
        


In [82]:
# Nairobi AE31 data
source = Path("data/level1/nrb")       # folder with .parquet files
target = Path("data/level2/nrb")      # where output should go
results = Path("results") / "aq_conference_nairobi_2025"
instrument_name = "ae31"

# aggregate data
df_ae31, path_ae31 = aggregate_data(
    source=source,       # folder with .parquet files
    target=target,      # where output should go
    instrument_name=instrument_name,
    freq="hourly",
    statistics="mean"
)
df_ae31.write_parquet(path_ae31)

# plot timeseries
# channels_and_colors = {
#     "UV370": "#8a2be2",  # Violet (BlueViolet)
#     "B470":  "#0000ff",  # Blue
#     "G520":  "#00cc00",  # Green
#     "Y590":  "#ffcc00",  # Yellow
#     "R660":  "#ff3300",  # Orange-Red
#     "IR880": "#800000",  # Dark Red (highlighted)
#     "IR950": "#a52a2a",  # Brownish IR
# }
# Define channels and corresponding wavelengths (in nm)
channels = ["UV370", "B470", "G520", "Y590", "R660", "IR880", "IR950"]
wavelengths = [370, 470, 520, 590, 660, 880, 950]

# Generate colors
colors = _wavelength_to_color(wavelengths)

# Build dictionary
channels_and_colors = dict(zip(channels, colors))

plot_time_series(df_ae31, channels_and_colors, highlight_channel="IR880", output_path=results / "ae31_timeseries.png", figsize=(10, 5), plot_type="scatter")

# plot diurnal variability
plot_diurnal_by_wavelength(df_ae31, channels_and_colors, output_path=results / "ae31_diurnal_cycle_box.png", plot_type="box", figsize=(10, 5))
# plot_diurnal_by_wavelength(df, output_path=results / "ae31_diurnal_cycle_violin.png", plot_type="violin")
# plot_diurnal_by_wavelength(df, output_path=results / "ae31_diurnal_cycle_swarm.png", plot_type="swarm")  # UserWarning: 83.9% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot. warnings.warn(msg, UserWarning)


Processing data/level1/nrb/2024/08/ae31.parquet ..
Processing data/level1/nrb/2024/09/ae31.parquet ..
Processing data/level1/nrb/2024/10/ae31.parquet ..
Processing data/level1/nrb/2024/11/ae31.parquet ..
Processing data/level1/nrb/2024/12/ae31.parquet ..
Processing data/level1/nrb/2025/01/ae31.parquet ..
Processing data/level1/nrb/2025/02/ae31.parquet ..
Processing data/level1/nrb/2025/03/ae31.parquet ..
Processing data/level1/nrb/2025/04/ae31.parquet ..
Processing data/level1/nrb/2025/05/ae31.parquet ..
Processing data/level1/nrb/2025/06/ae31.parquet ..
Processing data/level1/nrb/2025/07/ae31.parquet ..


/tmp/ipykernel_645372/2421933787.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("rainbow")


✅ Time series scatter plot saved to results/aq_conference_nairobi_2025/ae31_timeseries.png
✅ Diurnal box plot saved to results/aq_conference_nairobi_2025/ae31_diurnal_cycle_box.png


In [92]:
# Nairobi Fidas data
source = Path("data/level1/nrb")       # folder with .parquet files
target = Path("data/level2/nrb")      # where output should go
results = Path("results") / "aq_conference_nairobi_2025"
instrument_name = "fidas"
map = {'60': "Cn [1/cm³]",
        '61': "PM1 [mg/m³]",
        '62': "PM2.5 [mg/m³]",
        '63': "PM4 [mg/m³]",
        '64': "PM10 [mg/m³]",
        '65': "PMtotal [mg/m³]",
}

# aggregate data
df_fidas, path_fidas = aggregate_data(
    source=source,       # folder with .parquet files
    target=target,      # where output should go
    instrument_name=instrument_name,
    freq="hourly",
    extract_cols=map.keys(),
    statistics="mean",
)
df_fidas = df_fidas.rename({old: new for old, new in map.items() if old in df_fidas.columns})

df_fidas.write_parquet(path_fidas)

Processing data/level1/nrb/2025/05/fidas.parquet ..
Processing data/level1/nrb/2025/06/fidas.parquet ..


In [113]:
# plot timeseries
channels_and_colors = {
    "Cn [1/cm³]": "#1f77b4",  # Blue       — for number concentration (distinct from mass)
    "PM1 [mg/m³]": "#2ca02c",  # Green      — PM1
    "PM2.5 [mg/m³]": "#ff7f0e",  # Orange     — PM2.5
    "PM4 [mg/m³]": "#d62728",  # Red        — PM4
    "PM10 [mg/m³]": "#9467bd",  # Purple     — PM10
    "PMtotal [mg/m³]": "#8c564b",  # Brown      — PMtotal
}

plot_time_series(df_fidas, 
                 channels_and_colors=channels_and_colors, 
                 highlight_channel='Cn [1/cm³]', 
                 plot_type="line", 
                 output_path=results / "fidas_timeseries.png", 
                 figsize=(10, 5),
                 title="Fidas hourly data (Dagoretti Corner, NRB)",
                 ylabel=["Aerosol particle mass concentration [mg/m3]", "Aerosol particle number concentration [1/cm3]"],
                 which_y_axes=(2, 1, 1, 1, 1, 1),
                 legend_loc='upper left')


✅ Time series line plot saved to results/aq_conference_nairobi_2025/fidas_timeseries.png
